# TacticalGuard-LLM: Results Analysis
## MILCOM 2026 — Adversarial Robustness of LLM Cyber-Defense Agents

This notebook loads the 6 experimental condition logs and generates publication-ready figures.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path
from collections import defaultdict

from src.benchmark.logger import load_jsonl
from src.benchmark.metrics import Scorecard

# Military-style dark theme
sns.set_theme(style='darkgrid', context='paper', font_scale=1.2)
MILITARY_PALETTE = ['#c8a951', '#8fbc5f', '#5f9ea0', '#cd853f', '#8b6914', '#556b2f']
mpl.rcParams['figure.facecolor'] = '#1a1a2e'
mpl.rcParams['axes.facecolor'] = '#16213e'
mpl.rcParams['axes.edgecolor'] = '#c8a951'
mpl.rcParams['axes.labelcolor'] = '#e0e0e0'
mpl.rcParams['xtick.color'] = '#e0e0e0'
mpl.rcParams['ytick.color'] = '#e0e0e0'
mpl.rcParams['text.color'] = '#e0e0e0'
mpl.rcParams['grid.color'] = '#2a2a4a'
mpl.rcParams['legend.facecolor'] = '#1a1a2e'
mpl.rcParams['legend.edgecolor'] = '#c8a951'

os.makedirs('../results/figures', exist_ok=True)
FIGURES_DIR = Path('../results/figures')
print('Setup complete.')

In [ ]:
# Cell 1: Load all 6 condition logs
CONDITIONS = [
    'A_baseline', 'B_filter_only', 'C_filter_provenance',
    'D_defense_full', 'E_adaptive_whitebox', 'F_cross_model'
]
CONDITION_LABELS = [
    'A: Baseline', 'B: Filter Only', 'C: Filter+Provenance',
    'D: Full Defense', 'E: Adaptive (WB)', 'F: Cross-Model'
]

logs = {}
results = {}
sc = Scorecard()

for cond in CONDITIONS:
    log_path = f'../results/condition_{cond}_logs.jsonl'
    scorecard_path = f'../results/condition_{cond}_scorecard.json'
    
    if Path(log_path).exists():
        logs[cond] = load_jsonl(log_path)
        print(f'{cond}: {len(logs[cond])} steps loaded')
    else:
        print(f'{cond}: log file not found — using synthetic data for demo')
        # Synthetic demo data
        logs[cond] = []
    
    if Path(scorecard_path).exists():
        with open(scorecard_path) as f:
            results[cond] = json.load(f)
    elif logs.get(cond):
        results[cond] = sc.compute(logs[cond])
    else:
        # Synthetic placeholder results for demo
        import random
        rng = random.Random(hash(cond) % 2**32)
        results[cond] = {
            'CAR': rng.uniform(0.05, 0.35),
            'MTTF': rng.uniform(10, 45),
            'CatchRate': rng.uniform(0.6, 0.95),
            'FPR': rng.uniform(0.02, 0.15),
            'mean_latency_ms': rng.uniform(30, 200),
            'DST': rng.uniform(1.5, 3.0),
            'AER': rng.uniform(0.1, 0.45),
            'total_steps': 150,
        }

df = pd.DataFrame(results).T
df.index = CONDITION_LABELS
print('\nResults DataFrame:')
display(df[['CAR', 'MTTF', 'CatchRate', 'FPR', 'mean_latency_ms', 'AER']].round(4))

In [ ]:
# Cell 2: CAR Comparison Bar Chart
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    range(len(CONDITIONS)), 
    [results[c]['CAR'] for c in CONDITIONS],
    color=MILITARY_PALETTE,
    edgecolor='#c8a951',
    linewidth=0.8,
    alpha=0.85
)

for bar, val in zip(bars, [results[c]['CAR'] for c in CONDITIONS]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, color='#e0e0e0')

ax.set_xticks(range(len(CONDITIONS)))
ax.set_xticklabels(CONDITION_LABELS, rotation=20, ha='right')
ax.set_ylabel('Catastrophic Action Rate (CAR) ↓', color='#e0e0e0')
ax.set_title('CAR Across 6 Experimental Conditions\nTacticalGuard-LLM MILCOM 2026', 
             color='#c8a951', fontweight='bold')
ax.set_ylim(0, max(results[c]['CAR'] for c in CONDITIONS) * 1.3)

# Highlight best (lowest CAR)
best_idx = np.argmin([results[c]['CAR'] for c in CONDITIONS])
bars[best_idx].set_edgecolor('#8fbc5f')
bars[best_idx].set_linewidth(2.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_car_comparison.pdf', dpi=300, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.savefig(FIGURES_DIR / 'fig3_car_comparison.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Fig 3 saved.')

In [ ]:
# Cell 3: MTTF Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MTTF bars
mttf_vals = [results[c].get('MTTF', 0) for c in CONDITIONS]
axes[0].barh(CONDITION_LABELS, mttf_vals, color=MILITARY_PALETTE,
             edgecolor='#c8a951', linewidth=0.8, alpha=0.85)
axes[0].set_xlabel('Mean Time To First Failure (steps) ↑', color='#e0e0e0')
axes[0].set_title('MTTF — Defense Survival Duration', color='#c8a951', fontweight='bold')
for i, v in enumerate(mttf_vals):
    axes[0].text(v + 0.3, i, f'{v:.1f}', va='center', fontsize=9)

# CatchRate vs FPR scatter
catch = [results[c].get('CatchRate', 0) for c in CONDITIONS]
fpr = [results[c].get('FPR', 0) for c in CONDITIONS]
scatter = axes[1].scatter(fpr, catch, c=MILITARY_PALETTE, s=150, 
                          edgecolors='white', linewidth=1.5, zorder=5)
for i, (x, y, label) in enumerate(zip(fpr, catch, CONDITION_LABELS)):
    axes[1].annotate(label, (x, y), textcoords='offset points', 
                     xytext=(5, 5), fontsize=7.5, color='#e0e0e0')
axes[1].set_xlabel('False Positive Rate (FPR) ↓', color='#e0e0e0')
axes[1].set_ylabel('Catch Rate ↑', color='#e0e0e0')
axes[1].set_title('CatchRate vs FPR Trade-off', color='#c8a951', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mttf_catchrate.pdf', dpi=300, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.savefig(FIGURES_DIR / 'mttf_catchrate.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('MTTF figure saved.')

In [ ]:
# Cell 4: Multi-step chain phase analysis (Fig 4)
# Analyze which phase the defense first fails in (DST metric)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Phase action distribution from D_defense_full logs
phase_actions = {1: defaultdict(int), 2: defaultdict(int), 3: defaultdict(int)}

if logs.get('D_defense_full'):
    for step in logs['D_defense_full']:
        phase = step.get('attack_phase')
        action = step.get('action_parsed', 'Monitor')
        if phase in (1, 2, 3):
            phase_actions[phase][action] += 1
else:
    # Synthetic demo data
    from src.env.action_space import BLUE_ACTIONS
    import random
    rng = random.Random(42)
    for phase in (1, 2, 3):
        for action in BLUE_ACTIONS:
            phase_actions[phase][action] = rng.randint(0, 30)

# Stacked bar chart: action distribution per phase
actions = ['Monitor', 'Analyse', 'Remove', 'Restore', 'DeployDecoy', 'BlockTraffic', 'AllowTraffic']
phase_labels = ['Phase 1\n(Erosion)', 'Phase 2\n(Normalization)', 'Phase 3\n(Strike)']
x = np.arange(len(phase_labels))
width = 0.11

for i, action in enumerate(actions):
    vals = [phase_actions[p].get(action, 0) for p in (1, 2, 3)]
    axes[0].bar(x + i * width, vals, width, label=action,
                color=plt.cm.Set2(i / len(actions)), alpha=0.85)

axes[0].set_xticks(x + width * len(actions) / 2)
axes[0].set_xticklabels(phase_labels)
axes[0].set_ylabel('Action Count', color='#e0e0e0')
axes[0].set_title('Action Distribution per Attack Phase\n(Multi-Step Chain)', 
                   color='#c8a951', fontweight='bold')
axes[0].legend(fontsize=7, loc='upper right')

# DST distribution
dst_vals = [results[c].get('DST') for c in CONDITIONS if results[c].get('DST') is not None]
dst_conds = [CONDITION_LABELS[i] for i, c in enumerate(CONDITIONS) 
             if results[c].get('DST') is not None]
if dst_vals:
    axes[1].bar(dst_conds, dst_vals, color='#c8a951', edgecolor='white', 
                linewidth=0.8, alpha=0.85)
    axes[1].set_ylabel('Defense Survival Time (phase) ↑', color='#e0e0e0')
    axes[1].set_title('DST: Which Phase Defeats the Defense?', 
                      color='#c8a951', fontweight='bold')
    axes[1].set_yticks([1, 2, 3])
    axes[1].set_yticklabels(['Phase 1\n(Erosion)', 'Phase 2\n(Normal.)', 'Phase 3\n(Strike)'])
else:
    axes[1].text(0.5, 0.5, 'DST: No multi-step chain data\n(run Condition D with 50 episodes)',
                ha='center', va='center', transform=axes[1].transAxes, color='#e0e0e0')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_phase_analysis.pdf', dpi=300, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.savefig(FIGURES_DIR / 'fig4_phase_analysis.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Fig 4 saved.')

In [ ]:
# Cell 5: Adaptive attacker evasion rate — whitebox vs blackbox
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# AER comparison
aer_vals = [results[c].get('AER', 0) for c in CONDITIONS]
colors = ['#556b2f' if 'adaptive' not in c.lower() else '#cd853f' for c in CONDITIONS]

axes[0].bar(CONDITION_LABELS, aer_vals, color=colors, edgecolor='#c8a951',
            linewidth=0.8, alpha=0.85)
axes[0].set_ylabel('Adaptive Evasion Rate (AER) ↓', color='#e0e0e0')
axes[0].set_title('AER: Fraction of Attacks Evading Filter', 
                   color='#c8a951', fontweight='bold')
axes[0].tick_params(axis='x', rotation=25)
for i, v in enumerate(aer_vals):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

# Whitebox vs Blackbox (if E_adaptive data available)
strategies = ['synonym_replace', 'paraphrase', 'whitespace_inject', 'numeric_perturb']
wb_rates = [0.38, 0.29, 0.22, 0.31]  # placeholder — replace with actual from logs
bb_rates = [0.21, 0.18, 0.15, 0.19]  # placeholder

x = np.arange(len(strategies))
w = 0.35
axes[1].bar(x - w/2, wb_rates, w, label='White-box', color='#c8a951', alpha=0.85,
            edgecolor='white', linewidth=0.8)
axes[1].bar(x + w/2, bb_rates, w, label='Black-box', color='#5f9ea0', alpha=0.85,
            edgecolor='white', linewidth=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Synonym', 'Paraphrase', 'Whitespace', 'Numeric'], rotation=15)
axes[1].set_ylabel('Evasion Rate per Strategy ↓', color='#e0e0e0')
axes[1].set_title('Mutation Strategy Evasion Rates\nWhite-box vs Black-box', 
                   color='#c8a951', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'adaptive_evasion.pdf', dpi=300, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.savefig(FIGURES_DIR / 'adaptive_evasion.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Adaptive evasion figure saved.')

In [ ]:
# Cell 6: Cross-model transfer heatmap
# Shows how attacks optimized for LLaMA transfer to GPT-4o-mini

fig, ax = plt.subplots(figsize=(8, 5))

attack_types = ['obs_poison', 'comm_poison', 'reward_hack', 'prompt_inject', 'multi_step_chain']
models = ['LLaMA-3.1-8B', 'GPT-4o-mini', 'Gemini-Flash']

# CAR under each attack for each model (placeholder — replace with actual results)
transfer_data = np.array([
    [0.18, 0.15, 0.16],  # obs_poison
    [0.12, 0.10, 0.11],  # comm_poison  
    [0.09, 0.07, 0.08],  # reward_hack
    [0.22, 0.19, 0.20],  # prompt_inject
    [0.31, 0.27, 0.29],  # multi_step_chain
])

im = ax.imshow(transfer_data, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.4)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('CAR (Catastrophic Action Rate)', color='#e0e0e0')
cbar.ax.yaxis.set_tick_params(color='#e0e0e0')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#e0e0e0')

ax.set_xticks(range(len(models)))
ax.set_xticklabels(models)
ax.set_yticks(range(len(attack_types)))
ax.set_yticklabels(attack_types)
ax.set_title('Cross-Model Transfer: Attack CAR Across Model Families', 
             color='#c8a951', fontweight='bold')

for i in range(len(attack_types)):
    for j in range(len(models)):
        ax.text(j, i, f'{transfer_data[i,j]:.2f}', ha='center', va='center',
                fontsize=10, color='black' if transfer_data[i,j] < 0.25 else 'white')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cross_model_transfer.pdf', dpi=300, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.savefig(FIGURES_DIR / 'cross_model_transfer.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Cross-model transfer heatmap saved.')

In [ ]:
# Cell 7: Print full LaTeX scorecard table
sc = Scorecard()
latex = sc.generate_latex_table(results)
print('=== LaTeX Scorecard Table ===')
print(latex)

os.makedirs('../paper/tables', exist_ok=True)
with open('../paper/tables/results_table.tex', 'w') as f:
    f.write(latex)
print('\nSaved to ../paper/tables/results_table.tex')